# Results for model: microsoft/phi-3-medium-128k-instruct

In [1]:
import polars as pl
import pandas as pd
import xgboost as xgb
from typing import List
from sklearn.model_selection import train_test_split


# Custom function to calculate top quantile across rolling batches
def calculate_top_quantile(data, feature_col, responder_col, quantile=0.85, rolling_window=30):
    grouped = data.group_by('date_id').agg(
        f"{feature_col}_quantile={quantile}".format(quantile=quantile)
    )
    grouped['feature_col'] = grouped[f"{feature_col}_quantile={quantile}"].expanding_right(min_periods=1).median()
    grouped = grouped.fill_null(.0)
    data = data.merge(grouped).drop(columns=[f"{feature_col}_quantile={quantile}"])
    return data


def solve(path: str) -> List[str]:
    # Load the 'train.parquet' dataset
    df = pl.read_parquet(path).pipe(pd.DataFrame)

    # Feature Engineering
    # Calculate the rolling global top 15% of 'feature_00' relative to 'responder_6'
    df = calculate_top_quantile(df, 'feature_00', 'responder_6')

    # The feature to be used for prediction
    target_col = 'responder_6'

    # Extract features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Splitting data
    info = y.describe()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

    # Fit an XGBoost Regressor
    model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
    model.fit(X_train.to_pandas(), y_train, early_stopping_rounds=30, eval_set=[(X_test.to_pandas(), y_test)])

    return []  # empty response as per requirements


if __name__ == '__main__':
    solve(path='./jane_street_data/train.parquet')

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet